# 09. Bounded Foundry orchestration workbench

This notebook explores the second optional course-facing capability: a separate no-tools proposal-drafter role that consumes only an accepted evidence-analyst response. It does not turn the first agent into a hidden multi-agent system. The local evaluator still owns game outcomes, audits, target binding, evidence writing, and policy authority.

**Autonomy envelope:** define and guard the handoff locally. Do not deploy a second agent, modify the existing agent, invoke a model, add tools/connectors, or create a workflow resource in this notebook. Those are separately reviewed Foundry actions.

- [ ] Review whether a second proposal-drafter agent is needed for the submission.
- [ ] Review the handoff contract and its refusal cases before any deployment design.


## 1. Keep the role boundary explicit

The evidence analyst explains a packet. The optional proposal drafter may turn that explanation into a bounded *proposal record* for human review. It may not decide that a candidate won, change a source file, invoke the simulator, or expand the approved envelope. Passing data between roles is not permission to perform the next action.

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass(frozen=True)
class AnalystHandoff:
    """Only the accepted analyst facts permitted into the next role."""
    packet_sha256: str
    assessment: str
    citations: tuple[dict[str, str], ...]
    limitations: tuple[str, ...]
    review_status: str

def build_analyst_handoff(response: dict[str, Any]) -> AnalystHandoff:
    required = {"analyst_contract_version", "packet_sha256", "assessment", "citations", "limitations", "review_status"}
    if set(response) != required or response.get("review_status") != "pending_human_review":
        raise ValueError("proposal handoff requires the exact pending-review analyst response")
    if not response["citations"] or not response["limitations"]:
        raise ValueError("proposal handoff requires citations and limitations")
    return AnalystHandoff(response["packet_sha256"], response["assessment"], tuple(response["citations"]), tuple(response["limitations"]), response["review_status"])

ANALYST_RESPONSE = {
    "analyst_contract_version": 1, "packet_sha256": "c" * 64,
    "assessment": "The supplied packet is descriptive and does not establish a recommendation.",
    "citations": [{"packet_path": "aggregates[0]", "claim": "candidate 10: mean paired delta 0 across 4 matches"}],
    "limitations": ["descriptive fixture only", "no source-policy mutation is authorized"],
    "review_status": "pending_human_review",
}
HANDOFF = build_analyst_handoff(ANALYST_RESPONSE)


In [ ]:
from IPython.display import Markdown, display
display(Markdown(f"""### ✅ Analyst-to-drafter handoff

| Fact | Value |
|:---|:---|
| Packet binding | `{HANDOFF.packet_sha256[:12]}…` |
| Input role | evidence analyst |
| Input status | **{HANDOFF.review_status}** |
| Citations / limitations | **{len(HANDOFF.citations)} / {len(HANDOFF.limitations)}** |
| Simulator / source authority | **none** |
| Second agent deployment | **not performed** |

> The handoff is a bounded data contract, not an approval to run a proposal.
"""))


## 2. Constrained proposal-drafter instruction and output

The optional role is intentionally less authoritative than the local evaluator. Its output is a proposed next question, not a decision. Every proposal must carry its hypothesis, one bounded candidate envelope, success/failure criteria, limitations, citations, and `pending_human_review`.

In [ ]:
PROPOSAL_DRAFTER_INSTRUCTIONS = """You are a bounded proposal drafter. Use only the accepted analyst handoff.
Draft one possible next experiment question with one parameter and only
values already named by the owner or packet. Do not rank candidates, infer
game outcomes, call tools, edit source, invoke a simulator, write evidence,
or approve anything. Return JSON with hypothesis, candidate_values,
success_criteria, failure_criteria, citations, limitations, and
review_status=pending_human_review."""

PROPOSAL_SCHEMA = {
    "hypothesis": "one falsifiable question",
    "candidate_values": ["owner-declared values only"],
    "success_criteria": ["measurable criterion"],
    "failure_criteria": ["audit or interpretation stop"],
    "citations": ["exact handoff citation"],
    "limitations": ["copied handoff limits"],
    "review_status": "pending_human_review",
}

def validate_proposal_shape(proposal: dict[str, object]) -> dict[str, object]:
    if set(proposal) != set(PROPOSAL_SCHEMA):
        raise ValueError("proposal contains fields outside the bounded drafter contract")
    if proposal["review_status"] != "pending_human_review":
        raise ValueError("proposal must remain pending human review")
    if not proposal["candidate_values"] or not proposal["success_criteria"]:
        raise ValueError("proposal needs a bounded value list and success criteria")
    return proposal

PROPOSAL = validate_proposal_shape({
    "hypothesis": "A separately declared value may resolve the remaining uncertainty under a new approved contract.",
    "candidate_values": ["owner must declare before execution"],
    "success_criteria": ["all declared matches pass clean audit and meet the new contract"],
    "failure_criteria": ["any audit mismatch, undeclared value, or scope expansion"],
    "citations": [HANDOFF.citations[0]],
    "limitations": list(HANDOFF.limitations),
    "review_status": "pending_human_review",
})


In [ ]:
display(Markdown("""### Proposed optional orchestration role

| Fact | Value |
|:---|:---|
| Role | proposal drafter |
| Input | accepted analyst handoff only |
| Output | one bounded proposal record |
| Tools / simulator / source access | **none** |
| Human decision | approve, revise, or reject a new spec |
| Deployment | **not performed** |
| Review state | **pending_human_review** |

- [ ] Decide whether this optional second role is worth deploying for the course submission.

> This prototype demonstrates separation of roles; it is not a multi-agent deployment.
"""))


## 3. Guard the orchestration boundary

A proposal that tries to smuggle in a tool request is rejected before a future agent invocation. This is the same principle as the analyst response gate: structure and authority are deterministic; language generation is not authority.

In [ ]:
unsafe_proposal = dict(PROPOSAL)
unsafe_proposal["tool_request"] = "run simulator"
try:
    validate_proposal_shape(unsafe_proposal)
except ValueError as exc:
    ORCHESTRATION_GUARD = {"status": "rejected_before_invocation", "reason": str(exc)}
else:
    raise AssertionError("orchestration guard accepted a tool request")


In [ ]:
display(Markdown(f"""### ✅ Orchestration guard

| Guard | Result |
|:---|:---|
| Tool request added to proposal | **{ORCHESTRATION_GUARD['status']}** |
| Existing agent changed | **no** |
| New agent deployed | **no** |
| Human approval still required | **yes** |

> The optional role cannot turn a proposal into an execution command.
"""))
